# HY-489 Lab 5 — De-identification and privacy

**Friday 27 November 2026 · 2 hours · GRADED, 8% of the final mark**
**Deadline Thursday 3 December 2026, 23:59**

Name(s): _______________________

Read `HANDOUT.md` alongside this. The handout has the reasoning and the full
rubric; this notebook is the session.

**55 of the 100 marks are for what you write down**, in
`notebooks/deident_report.md`, not for what your code matched. The handout says
which five sections and what each one is worth.

**The rule, again.** Nothing real goes into this notebook, into `out/`, onto a
public DICOM server, or into this repository. Not your own images, not a
friend's, not an extract from a placement. The archive you are about to
de-identify is synthetic and `../../lab04/gen_archive.py` is how it was made.
Looking up your own record, or a relative's, is the most common reason people
are dismissed from health care jobs, and running a de-identification script
over real data to see whether it works is the same act with a better excuse.

In [ ]:
import os, sys, csv, json
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import deident, pixels, profile, pseudonym, verify

ARCHIVE   = "../../lab04/data/archive"
INVENTORY = "../../lab04/notebooks/inventory.csv"
RELEASE   = "../out/archive"
OUT       = "../out"
KEY       = "../key"
SCRATCH   = "../scratch"

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 44)
print("pydicom", __import__("pydicom").__version__)
print("files under the archive:", sum(len(f) for _, _, f in os.walk(ARCHIVE)))
print("Lab 4 inventory present:", os.path.isfile(INVENTORY))

---
## Part A · The profile

### Task 1 — read the table before you use it

`src/profile.py` is the subset of DICOM PS3.15 Table E.1-1 this lab works
from. Print it, then answer the three questions below.

In [ ]:
profile.describe()

In [ ]:
for code_, meaning in sorted(profile.ACTION_CODES.items()):
    print("%-7s %s" % (code_, meaning))
print()
print("applied actions across the whole profile:", profile.action_counts())

**Your answers.**

1. `PatientID` is `Z/D` and `SOPInstanceUID` is `U`. Both are identifiers.
   Why is one of them allowed to become empty and the other not?

   _write here_

2. Find the six rows whose `source` is `local`. What is the difference between
   an attribute the standard says `K` and an attribute the standard does not
   mention?

   _write here_

3. `profile.RESOLUTION` turns every combination code into one action. Name one
   row where you would have resolved it differently and say what that costs.

   _write here_

### Task 2 — the archive against the table

Which attributes does this archive actually contain, and which of them does the
profile say anything about?

In [ ]:
import pydicom

seen = set()
for rel in verify.walk_relative(ARCHIVE):
    try:
        ds = pydicom.dcmread(os.path.join(ARCHIVE, rel), stop_before_pixels=True)
    except Exception:
        continue
    for element in ds:
        if element.keyword:
            seen.add(element.keyword)

covered, uncovered = profile.coverage(sorted(seen))
print("distinct attributes in the archive  %d" % len(seen))
print("  covered by the profile            %d" % len(covered))
print("  not covered                       %d" % len(uncovered))
print()
print("\n".join("  " + k for k in uncovered))

**Your answer.** Every attribute on that list is released exactly as it is.
Pick the two you are least comfortable with and say what an attacker would do
with them.

_write here_

### Task 3 — `patient_pseudonym`, in `src/pseudonym.py`

`naive_pseudonym` is written for you and it is the thing to beat. Fill the stub,
then run the cell below. Part C attacks whatever you wrote.

In [ ]:
SECRET = pseudonym.make_secret("change this phrase and do not put it in the report")

print("naive :", pseudonym.naive_pseudonym("PAGNI100001"))
print("yours :", pseudonym.patient_pseudonym("PAGNI100001", SECRET))
print("again :", pseudonym.patient_pseudonym("PAGNI100001", SECRET))
print("other secret:", pseudonym.patient_pseudonym("PAGNI100001", "something else"))
print("other patient:", pseudonym.patient_pseudonym("PAGNI100138", SECRET))

### Task 4 — `remap_uid`, in `src/pseudonym.py`

One original UID, one replacement, everywhere it appears.

In [ ]:
uid_map = {}
a = pseudonym.remap_uid("1.2.840.113619.2.55.3.1", uid_map, SECRET)
b = pseudonym.remap_uid("1.2.840.113619.2.55.3.1", uid_map, SECRET)
c = pseudonym.remap_uid("1.2.840.113619.2.55.3.2", uid_map, SECRET)
print("same original twice :", a == b)
print("two originals differ:", a != c)
print("under the 2.25 arc  :", str(a).startswith("2.25."))
print("length              :", len(str(a)), "of a maximum 64")
print(a)

### Task 5 — `apply_profile`, in `src/deident.py`

The stub walks `profile.PROFILE`, asks `ctx.effective_action(entry)` what to do
and does it. Fill it, then run it on one instance and read the before and after
side by side before you run it on 84.

In [ ]:
ctx = deident.Context(SECRET)
print("options claimed:", sorted(ctx.options))

one = "incoming_2026_03/img_0001.dcm"
ds = pydicom.dcmread(os.path.join(ARCHIVE, one))
ctx.current_patient = str(ds.PatientID)

before = {e.keyword: str(ds.get(e.keyword, "<absent>")) for e in profile.PROFILE}
touched = deident.apply_profile(ds, ctx, path=one)
after = {e.keyword: str(ds.get(e.keyword, "<absent>")) for e in profile.PROFILE}

print("attributes acted on:", touched)
print()
print("%-28s %-34s %s" % ("keyword", "before", "after"))
for e in profile.PROFILE:
    if before[e.keyword] == "<absent>" and after[e.keyword] == "<absent>":
        continue
    print("%-28s %-34s %s" % (e.keyword, before[e.keyword][:34],
                              after[e.keyword][:34]))

**Your answer.** One of those rows changed and should not have, or did not
change and should have, depending on how you resolved the combination codes.
Which one, and what did you decide?

_write here_

### Checkpoint 1 — 0:50

De-identify the whole archive. The list of instances comes from the Lab 4
inventory. If you do not have `inventory.csv`, the handout says how to
regenerate it; passing `paths=None` walks the tree instead, and the difference
between the two lists is itself a finding.

In [ ]:
if os.path.isfile(INVENTORY):
    with open(INVENTORY, newline="", encoding="utf-8") as fh:
        paths = [r["path"] for r in csv.DictReader(fh)]
    print("instance list from the Lab 4 inventory:", len(paths))
else:
    paths = None
    print("no inventory.csv; walking the archive instead")

ctx = deident.Context(SECRET)
rows, summary = deident.run_archive(ARCHIVE, RELEASE, ctx, paths=paths)
deident.summarise(summary)

**Six numbers to be able to say out loud.** Instances requested, instances
written, pseudonyms, UIDs remapped, and the two lists at the bottom. One file
on disk is not in your instance list. Say what it is and why that is correct.

_write here_

---
## Part B · What the profile does not reach

### Task 1 — UID consistency

The profile said `U` four times. Check that it meant it.

In [ ]:
uids = verify.check_uids(ARCHIVE, RELEASE, ctx.uid_map)
print("originals mapped      ", uids["mapped"])
print("distinct replacements ", uids["distinct_new"])
print("problems              ", len(uids["problems"]))
for p in uids["problems"][:10]:
    print("   ", p)
print()
print("archive", uids["original"])
print("release", uids["released"])

**Your answer.** The release has one fewer distinct instance identifier than it
has files. That is Lab 4's duplicate `SOPInstanceUID`, still there. Should
de-identification have fixed it?

_write here_

### Task 2 — `blank_region`, in `src/pixels.py`

One object in this archive has the patient's name drawn into the pixels. The
detector is written for you. The blanking is yours.

In [ ]:
annotated = [r for r in rows if r["annotation"].startswith("blanked")]
for r in annotated:
    print(r["source"], "->", r["output"])
    print("   ", r["annotation"])

src = pydicom.dcmread(os.path.join(ARCHIVE, annotated[0]["source"]))
rel = pydicom.dcmread(os.path.join(RELEASE, annotated[0]["output"]))
a0, a1 = src.pixel_array, rel.pixel_array
box = pixels.find_annotation_band(a0)
print()
print("before:", pixels.band_report(a0, box))
print("after :", int((a0 != a1).sum()), "pixels differ")
print("after : the detector now finds", pixels.find_annotation_band(a1))
print("after : BurnedInAnnotation is", rel.get("BurnedInAnnotation"))

In [ ]:
# Look at it. A redaction you cannot see is a redaction you cannot check.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(a0, cmap="gray"); ax[0].set_title("archive")
ax[1].imshow(a1, cmap="gray"); ax[1].set_title("release")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

**Your answer.** The detector found one band in one image. Write the sentence
you would put in a release note about every other image in this archive.

_write here_

### Task 3 — the structured report

One instance has no pixels and carries free text. The Basic Profile's action on
`ContentSequence` is `D`.

In [ ]:
sr = [r for r in rows if r["modality"] == "SR"][0]
src = pydicom.dcmread(os.path.join(ARCHIVE, sr["source"]))
rel = pydicom.dcmread(os.path.join(RELEASE, sr["output"]))
print("archive text:", repr(str(src.ContentSequence[0].TextValue)))
print("release items:", len(rel.ContentSequence))
print("release text :", repr(str(rel.ContentSequence[0].get("TextValue", "<absent>"))))

In [ ]:
# The same archive with the Clean Structured Content Option claimed.
ctx_sr = deident.Context(SECRET, options=("BASIC", "CLEAN_PIXEL_DATA",
                                          "RETAIN_MODIFIED_DATES",
                                          "CLEAN_STRUCTURED_CONTENT"))
rows_sr, summary_sr = deident.run_archive(ARCHIVE, os.path.join(SCRATCH, "out_sr", "archive"), ctx_sr,
                                          paths=paths)
sr2 = [r for r in rows_sr if r["modality"] == "SR"][0]
rel2 = pydicom.dcmread(os.path.join(os.path.join(SCRATCH, "out_sr", "archive"), sr2["output"]))
print("text items changed by the clean:", sr2["cleaned_text_items"])
print("release text:", repr(str(rel2.ContentSequence[0].TextValue)))

**Your answer.** The clean changed nothing, because this report contains no
identifier. Say what that does and does not tell you, and say which of the two
releases you would ship.

_write here_

### Task 4 — `shift_dates`, in `src/deident.py`

This run claims the Retain Longitudinal Modified Dates Option, which is why
`StudyDate` survived Part A at all. Fill the stub, rerun Checkpoint 1, and then
check that the intervals inside a patient are the ones that were there.

In [ ]:
for key in sorted(ctx.patient_map):
    print("%-13s offset %+4d days   %s" % (key, ctx.date_offset(key),
                                           ctx.patient_map[key]))

In [ ]:
pairs = set()
for r in rows:
    if not r["output"]:
        continue
    s = pydicom.dcmread(os.path.join(ARCHIVE, r["source"]), stop_before_pixels=True)
    if str(s.get("PatientID", "")) != "PAGNI100001":
        continue
    o = pydicom.dcmread(os.path.join(RELEASE, r["output"]), stop_before_pixels=True)
    pairs.add((str(s.StudyDate), str(o.StudyDate)))

import datetime
def d(s): return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:8]))
pairs = sorted(pairs)
for b, a in pairs:
    print(b, "->", a)
if len(pairs) == 2:
    print("interval before", (d(pairs[1][0]) - d(pairs[0][0])).days, "days")
    print("interval after ", (d(pairs[1][1]) - d(pairs[0][1])).days, "days")

### Task 5 — the instance whose pixels are gone

Lab 4 found it. It is in your inventory and it has no image.

In [ ]:
broken = [r for r in rows if r["pixels_ok"] is False]
for r in broken:
    print(r["source"], "|", r["status"])
    print("   released as:", r["output"] or "NOT RELEASED")
    if r["output"]:
        o = pydicom.dcmread(os.path.join(RELEASE, r["output"]))
        print("   header complete, PixelData is", len(o.PixelData), "bytes")
        try:
            o.pixel_array
            print("   pixels decode in the release")
        except Exception as exc:
            print("   pixels still do not decode:", type(exc).__name__)

**Checkpoint 2 — 1:30.** On screen: the UID check with no problems, the two
pictures of the secondary capture, the two versions of the report, the date
intervals, and the broken instance.

**Your answer.** You released it, or you did not. Say which and say what the
other choice would have cost.

_write here_

---
## Part C · Prove it

This is Week 8's attack, pointed at what you made twenty minutes ago.

In [ ]:
needles, original_uids = verify.known_identifiers(ARCHIVE)
print("needles:", len(needles), "strings and", len(original_uids), "UIDs")
leaks = verify.scan_for_leaks(RELEASE, needles, original_uids)
print("hits:", len(leaks))
for h in leaks[:20]:
    print("  ", h["path"], h["keyword"], h["needle"])

### The hash trap

Week 8 said a hash is one-way only when the input cannot be guessed. A `PAGNI`
number is a fixed prefix and six digits.

In [ ]:
naive = {pid: pseudonym.naive_pseudonym(pid) for pid in ctx.patient_map}
recovered, tried = verify.attack_pseudonyms(set(naive.values()),
                                            pseudonym.naive_pseudonym)
print("unkeyed SHA-256: %d candidates tried, %d of %d recovered"
      % (tried, len(recovered), len(naive)))
for p, original in sorted(recovered.items())[:5]:
    print("   %-26s <- %s" % (p, original))

In [ ]:
released = verify.released_pseudonyms(RELEASE)

got_key, _ = verify.attack_pseudonyms(
    released, lambda v: pseudonym.patient_pseudonym(v, SECRET))
no_key, tried = verify.attack_pseudonyms(
    released, lambda v: pseudonym.patient_pseudonym(v, "not the secret"))

print("your scheme, attacker holds the key: %d of %d recovered"
      % (len(got_key), len(released)))
print("your scheme, attacker has no key   : %d of %d recovered, %d tried"
      % (len(no_key), len(released), tried))

**Your answer.** The second number is zero and the first is not. Write the
sentence that says what that buys and what it does not, with GDPR Recital 26 in
it.

_write here_

### The dates

The attacker holds the hospital's own worklist, which is the archive, and is
given the release.

In [ ]:
dates = verify.attack_dates(ARCHIVE, RELEASE)
print(dates)

ctx_full = deident.Context(SECRET, options=("BASIC", "CLEAN_PIXEL_DATA",
                                            "RETAIN_FULL_DATES"))
rows_full, _ = deident.run_archive(ARCHIVE, os.path.join(SCRATCH, "out_full", "archive"), ctx_full,
                                   paths=paths)
dates_full = verify.attack_dates(ARCHIVE, os.path.join(SCRATCH, "out_full", "archive"))
print()
print("dates shifted per patient: %(matched_uniquely)d of %(released_studies)d "
      "studies matched to exactly one archive study" % dates)
print("full dates retained      : %(matched_uniquely)d of %(released_studies)d "
      "studies matched to exactly one archive study" % dates_full)

### The shape

Take the dates away from the attacker entirely. They are left with each study's
modalities, how many series it has and how many instances. None of that is an
attribute and no option in PS3.15 removes any of it.

In [ ]:
shape = verify.attack_shape(ARCHIVE, RELEASE)
print(shape)
print()
print("%d of %d released studies match exactly one archive study on shape alone"
      % (shape["matched_uniquely"], shape["released_studies"]))

**Your answer.** This is the number that does not go down when you claim another
option. Write the sentence about it that belongs in section 5.

_write here_

### Checkpoint 3 — 1:55

In [ ]:
structure = verify.check_structure(RELEASE)
verify.report(leaks, uids, (no_key, tried), dates, structure, shape=shape)

In [ ]:
os.makedirs(OUT, exist_ok=True)
os.makedirs(KEY, exist_ok=True)

deident.save_manifest(rows, os.path.join(OUT, "manifest.csv"))
deident.save_crosswalk(rows, os.path.join(KEY, "crosswalk.csv"))
pseudonym.save_uid_map(ctx.uid_map, os.path.join(KEY, "uid_map.csv"))

# The link table needs the cohort column from Lab 4 Part C. Fill link_status
# and cohort_id from your Lab 4 reconciliation, or leave them blank and say in
# the report that you did.
link = [{"pseudonym": ctx.patient_map[pid], "patient_id": pid,
         "cohort_id": "", "link_status": "",
         "instances": sum(1 for r in rows if r["pseudonym"] == ctx.patient_map[pid]),
         "note": ""}
        for pid in sorted(ctx.patient_map)]
pseudonym.save_link_table(link, os.path.join(KEY, "link_table.csv"))

verify.to_json("audit.json", leaks=leaks, uids=uids,
               hash_attack={"tried": tried, "recovered": len(no_key),
                            "released": len(released)},
               dates=dates, dates_full=dates_full, shape=shape,
               structure=structure,
               summary=summary)
print("written")

---
## Wrap

Write `notebooks/deident_report.md` with exactly these five headings. The
handout says what each one is worth.

```
## 1. The profile
## 2. Identifiers and UIDs
## 3. The four decisions
## 4. What the profile did not reach
## 5. The audit, read back
```

Then, from `labs/lab05/`:

```bash
python3 check_submission.py
```

`../scratch/` holds the two experimental releases from Part B and Part C.
Delete it before you submit: one of them retains full dates and is the less
de-identified of the two things you made.

**Submit** `out/`, `key/`, `notebooks/` and `src/`. The release and the key are
submitted separately in a real project and together here, because the point of
the exercise is that you can show both and show that they are not the same
folder.